<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-08-tool-use-and-mcp/lesson-8.2-designing-tool-schemas/notebooks/exercises-8_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8.2: Designing Tool Schemas

*Module 8 · MCP, Tool Orchestration & Function Calling LIVE CLASS*

> The LLM reads your tool schema to decide WHEN to call it and WHAT arguments to pass. A vague schema = wrong tool calls. A precise schema = reliable execution. This lesson teaches schema design as an engineering discipline: JSON Schema fundamentals, Pydantic models for validation, description optimization tested with real LLMs, and the patterns that separate 60% accuracy from 95%.

`JSON Schema` · `Pydantic` · `Validation` · `Description Tuning` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-8.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Pydantic Models — Type-Safe Tool Schemas — `01_pydantic_tools.py`
2. Step 2 — Schema-to-Provider — Auto-Export for Gemini, OpenAI, Anthropic — `02_schema_export.py`
3. Step 3 — Description Optimization — The Words That Matter Most — `03_description_tuning.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai pydantic


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Pydantic Models — Type-Safe Tool Schemas

Define tool inputs with Pydantic. Get validation, documentation, and JSON Schema for free.

**`01_pydantic_tools.py`** · _Block 1 of 3_

PYDANTIC TOOL SCHEMAS — Type-safe, self-documenting — pip install pydantic


In [ ]:
# PYDANTIC TOOL SCHEMAS — Type-safe, self-documenting
# pip install pydantic
from pydantic import BaseModel, Field
from typing import Optional, Literal
import json

# ── Tool 1: Weather ──
class WeatherInput(BaseModel):
    city: str = Field(description="City name, e.g. 'Hyderabad' or 'Mumbai'")
    unit: Literal["celsius", "fahrenheit"] = Field(default="celsius", description="Temperature unit")

# ── Tool 2: Course search ──
class CourseSearchInput(BaseModel):
    topic: str = Field(description="Course topic: 'genai', 'python', 'cloud', 'fullstack'")
    max_price: Optional[int] = Field(default=None, description="Maximum price in INR")
    sort_by: Literal["price", "rating", "duration"] = Field(default="rating", description="Sort order")

# ── Tool 3: Booking ──
class BookingInput(BaseModel):
    course_id: str = Field(description="Course ID from search results, e.g. 'GENAI-001'")
    student_email: str = Field(description="Student's email address")
    payment_method: Literal["upi", "card", "emi"] = Field(description="Payment method")

# ── Auto-generate JSON Schema ──
print("Pydantic → JSON Schema:\n")
for name, model in [("WeatherInput",WeatherInput), ("CourseSearchInput",CourseSearchInput), ("BookingInput",BookingInput)]:
    schema = model.model_json_schema()
    print(f"  {name}:")
    print(f"    Properties: {list(schema.get('properties',{}).keys())}")
    print(f"    Required:   {schema.get('required',[])}\n")

# ── Validate input ──
try:
    valid = CourseSearchInput(topic="genai", max_price=15000)
    print(f"  Valid: {valid.model_dump()}")
except Exception as e:
    print(f"  Error: {e}")

try:
    bad = CourseSearchInput(topic="genai", sort_by="popularity")  # not in Literal!
except Exception as e:
    print(f"  Rejected: {str(e)[:80]}")

print(f"\n  Pydantic gives you: type checking, validation, JSON Schema, docs — all free.")


> **What just happened?** Three Pydantic models defined tool inputs with types, defaults, constraints ( Literal for enums), and descriptions. model_json_schema() auto-generated JSON Schema for any provider. Validation caught sort_by="popularity" because it’s not in the allowed values. Pydantic = single source of truth for: validation (runtime), documentation (schema), and LLM tool definition (JSON Schema). Write once, use everywhere.


### Step 2 · Schema-to-Provider — Auto-Export for Gemini, OpenAI, Anthropic

One Pydantic model, three provider formats. Auto-generated.

**`02_schema_export.py`** · _Block 2 of 3_

AUTO-EXPORT PYDANTIC SCHEMAS TO ALL PROVIDERS


In [ ]:
# AUTO-EXPORT PYDANTIC SCHEMAS TO ALL PROVIDERS
from pydantic import BaseModel, Field
from typing import Optional, Literal
import json, inspect

class ToolSchema:
    """Convert Pydantic model to any provider's tool format."""
    def __init__(self, name, description, input_model, func):
        self.name = name
        self.desc = description
        self.model = input_model
        self.func = func

    def to_openai(self):
        return {"type":"function", "function": {
            "name": self.name, "description": self.desc,
            "parameters": self.model.model_json_schema()}}

    def to_anthropic(self):
        return {"name": self.name, "description": self.desc,
                "input_schema": self.model.model_json_schema()}

    def to_gemini(self):
        return self.func  # Gemini uses the Python function directly

    def validate_and_call(self, args):
        validated = self.model(**args)
        return self.func(**validated.model_dump())

# ── Define tool ──
class SearchInput(BaseModel):
    query: str = Field(description="Search query")
    max_results: int = Field(default=5, ge=1, le=20, description="Max results 1-20")
    category: Optional[Literal["courses","blog","faq"]] = Field(default=None)

def search_knowledge(query, max_results=5, category=None):
    return {"results": [f"Result for '{query}'"], "count": max_results}

tool = ToolSchema("search_knowledge",
    "Search Netsetos knowledge base for courses, blog posts, or FAQ answers.",
    SearchInput, search_knowledge)

print("Schema Export:\n")
print(f"  OpenAI:    {json.dumps(tool.to_openai()['function']['name'])}")
print(f"  Anthropic: {json.dumps(tool.to_anthropic()['name'])}")
print(f"  Gemini:    {tool.to_gemini().__name__}\n")

# Validate + execute
result = tool.validate_and_call({"query": "GenAI course", "max_results": 3})
print(f"  Validated call: {result}")

try:
    tool.validate_and_call({"query": "test", "max_results": 100})  # exceeds le=20
except Exception as e:
    print(f"  Rejected: max_results=100 exceeds limit (max 20)")


### Step 3 · Description Optimization — The Words That Matter Most

The LLM reads descriptions to decide which tool to call. Better descriptions = better routing.

**`03_description_tuning.py`** · _Block 3 of 3_

DESCRIPTION OPTIMIZATION — Test which descriptions work best


In [ ]:
# DESCRIPTION OPTIMIZATION — Test which descriptions work best
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def test_tool_routing(tools, queries):
    """Test which tool the LLM picks for each query."""
    model = genai.GenerativeModel("gemini-2.0-flash", tools=tools)
    chat = model.start_chat()
    results = []
    for q in queries:
        try:
            resp = chat.send_message(q)
            # Check if a tool was called
            called = "direct"
            for part in resp.candidates[0].content.parts:
                if hasattr(part, 'function_call') and part.function_call:
                    called = part.function_call.name
            results.append((q, called))
        except:
            results.append((q, "error"))
    return results

# ── Bad descriptions ──
def search_v1(q: str) -> dict:
    """Search stuff."""  # vague!
    return {"r": q}
def booking_v1(id: str) -> dict:
    """Do booking."""  # vague!
    return {"r": id}

# ── Good descriptions ──
def search_v2(query: str, max_results: int = 5) -> dict:
    """Search Netsetos course catalog by topic, price, or keyword.
    Use when the user asks about available courses, pricing, or course details.
    Returns course names, prices, and ratings."""
    return {"results": []}
def booking_v2(course_id: str, email: str) -> dict:
    """Book a course for a student. Use ONLY when the user explicitly wants
    to enroll or purchase. Requires course_id from search results and student email."""
    return {"booked": True}

queries = [
    "What GenAI courses do you have?",
    "I want to enroll in the GenAI course",
    "How much does the Python course cost?",
    "Book me the cloud course, my email is [email protected]",
]

print("Description Optimization:\n")
print("  Testing with VAGUE descriptions:")
for q, tool in test_tool_routing([search_v1, booking_v1], queries):
    print(f"    '{q[:40]}...' → {tool}")

print(f"\n  Testing with PRECISE descriptions:")
for q, tool in test_tool_routing([search_v2, booking_v2], queries):
    print(f"    '{q[:40]}...' → {tool}")

print(f"\n  Key: add WHEN to use, WHAT it returns, and examples in description.")


> **What just happened?** Same 4 queries, two schema versions. Vague descriptions (“search stuff”) caused wrong routing — booking called for search queries. Precise descriptions (“Search course catalog... use when user asks about available courses”) routed correctly. Three things that fix 90% of routing errors: (1) say WHEN to use the tool, (2) say WHAT it returns, (3) add 1-2 example inputs in the description.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
